In [18]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from sklearn.linear_model import LinearRegression

class stkSeries():
    '''
    date format:%Y-%m-%d and end date is not reached;interval default :1h
    '''
    def __init__(self,code,start,end,interval='1h'):
        self.stk = code
        self.start = start
        self.end = end
        self.interval = interval
        self.data = self.get_series()

    def get_series(self):
        ticker = yf.Ticker(self.stk)
        stk = ticker.history(start=self.start,end=self.end,
        interval = self.interval,auto_adjust = True).drop(['Dividends','Stock Splits'],axis=1)
        return stk
    def r2_market(self,year=1,window=30):
        '''
        year: history range
        window: R2 calculate window
        '''
        #for history date 
        end_date = datetime.strptime(self.end,'%Y-%m-%d')
        start_date = end_date -relativedelta(years=year)
        start = start_date.strftime('%Y-%m-%d')##

        #get time series data
        stk = self.data['Close']#stock
        dji = stkSeries('^DJI',start = self.start,end=self.end).data['Close']#don johns
        sp500 = stkSeries('SPY',start = self.start,end=self.end).data['Close']#sp500
        nasdaq = stkSeries('^IXIC',start = self.start,end=self.end).data['Close']#nasdaq
        qqq = stkSeries('QQQ',start = self.start,end=self.end).data['Close']#qqq

        #history time series
        stk_h = stkSeries(self.stk,start = start, end = self.end).data['Close']
        dji_h = stkSeries('^DJI',start = start, end = self.end).data['Close']
        sp500_h = stkSeries('SPY',start = start, end = self.end).data['Close']
        nasdaq_h = stkSeries('^IXIC',start = start, end = self.end).data['Close']
        qqq_h = stkSeries('QQQ',start = start, end = self.end).data['Close']

        #calculate r2 function
        def lr_r2(respond, features):
            '''
            respond: series for stk 
            features: list of series of market
            '''
            df = pd.concat([respond] + features, axis=1).dropna()
            df = df.pct_change().dropna()
            y = df.iloc[:,0].to_numpy()
            X = df.iloc[:,1:].to_numpy()
            model = LinearRegression()
            model.fit(X, y)
            return model.score(X, y)
        
        def lr_r2_h(respond,features):
            '''
            respond: series for stk 
            features: list of series of market
            '''
            df = pd.concat([respond] + features, axis=1, join='inner').dropna()
            df.columns = ['y'] + [f'x{i+1}' for i in range(len(features))]

            df = df.pct_change().dropna()
            y = df['y'].to_numpy()
            X = df.drop(columns=['y']).to_numpy()
            n = len(y)
            r2_list = []
            model = LinearRegression()
            for i in range(window, n + 1):
                y_window = y[i - window:i]
                X_window = X[i - window:i]
                model.fit(X_window, y_window)
                r2 = model.score(X_window, y_window)
                r2_list.append(r2)

            return r2_list
        #current r2
        r2_dji = lr_r2(stk,[dji])
        r2_sp500 = lr_r2(stk,[sp500])
        r2_nasdaq = lr_r2(stk,[nasdaq])
        r2_qqq = lr_r2(stk,[qqq])
        r2_total = lr_r2(stk,[qqq,sp500])

        #history r2
        r2_dji_h = lr_r2_h(stk_h,[dji_h])
        r2_sp500_h = lr_r2_h(stk_h,[sp500_h])
        r2_nasdaq_h = lr_r2_h(stk_h,[nasdaq_h])
        r2_qqq_h = lr_r2_h(stk_h,[qqq_h])
        r2_total_h = lr_r2_h(stk_h,[qqq_h,sp500_h])

        #show
        df = pd.DataFrame({
        'dji': [
            r2_dji,
            np.mean(r2_dji_h),
            np.std(r2_dji_h)
        ],
        'sp500': [
            r2_sp500,
            np.mean(r2_sp500_h),
            np.std(r2_sp500_h)
        ],
        'nasdaq': [
            r2_nasdaq,
            np.mean(r2_nasdaq_h),
            np.std(r2_nasdaq_h)
        ],
        'qqq': [
            r2_qqq,
            np.mean(r2_qqq_h),
            np.std(r2_qqq_h)
        ],
        'total':[
            r2_total,
            np.mean(r2_total_h),
            np.std(r2_total_h)
        ],

        }, index=['current_r2', 'mean_r2_h', 'std_r2_h'])
        df.loc['percentile_h'] = [
            np.mean(np.array(r2_dji_h) <= r2_dji) * 100,
            np.mean(np.array(r2_sp500_h) <= r2_sp500) * 100,
            np.mean(np.array(r2_nasdaq_h) <= r2_nasdaq) * 100,
            np.mean(np.array(r2_qqq_h) <= r2_qqq) * 100,
            np.mean(np.array(r2_total_h) <= r2_total) * 100

        ]
        return df

In [11]:
ticker = yf.Ticker('^DJI')
stk = ticker.history(start='2025-01-01',end='2025-01-10',
        interval = '1h',auto_adjust = True).drop(['Dividends','Stock Splits'],axis=1)
# stk.index = pd.to_datetime(stk.index.date)
# stk

In [21]:
aapl = stkSeries('AAPL','2025-10-09','2025-10-11')
aapl.r2_market()

,dji,sp500,nasdaq,qqq,total
current_r2,0.892310,0.904566,0.882202,0.926875,0.927870
mean_r2_h,0.295640,0.368233,0.379055,0.369497,0.429817
std_r2_h,0.257601,0.278570,0.271512,0.268847,0.265074
percentile_h,98.360656,98.009368,97.658080,99.473068,98.653396
